In [17]:
# --- STEP 1: READ AND CHECK OUR FILE.TXT ---
file_name = 'test1_DataOutbound.txt'

with open(file_name, 'r', encoding='utf-8') as f:
    my_ips = [line.strip() for line in f if line.strip()]

print(f"Total IPs found: {len(my_ips)}")
print("-" * 30)

# 1. Show the Top 3
print("TOP 3 IPs:")
for ip in my_ips[:3]:
    print(f"  > {ip}")

print("-" * 30)

# 2. Show the Bottom 3
print("BOTTOM 3 IPs:")
for ip in my_ips[-3:]:
    print(f"  > {ip}")

Total IPs found: 3985
------------------------------
TOP 3 IPs:
  > 8.8.8.8
  > 8.8.4.4
  > 58.26.39.135
------------------------------
BOTTOM 3 IPs:
  > 2404:6800:4001:812::200a
  > 2a05:d018:91c:3200:2846:99fb:81b6:1e11
  > 2a05:d018:91c:3200:5e0d:21a9:26ca:90b5


In [ ]:
# --- STEP 2: READ AND CHECK CONNECTIVITY GITHUB FILE.TXT AND AWARE FOR ANY UPDATE OF THOSE FILES.TXT---

import requests

# The GitHub URL
github_url = "https://raw.githubusercontent.com/O-X-L/risk-db-lists/refs/heads/main/ip/kind_proxy.txt"

try:
    # Fetching the data
    response = requests.get(github_url, timeout=15)
    
    if response.status_code == 200:
        # Convert text to a list, remove empty lines, and SORT them
        # Sorting is important so the 'Top 3' are the lowest numbers (e.g. 1.x.x.x)
        github_ips = sorted([line.strip() for line in response.text.splitlines() if line.strip()])

        print(f"Total IPs found: {len(github_ips)}")
        print("-" * 30)

        # 1. Show the Top 3 from GitHub
        print("GITHUB TOP 3 IPs:")
        for ip in github_ips[:3]:
            print(f"  > {ip}")

        print("-" * 30)

        # 2. Show the Bottom 3 from GitHub
        print("GITHUB BOTTOM 3 IPs:")
        for ip in github_ips[-3:]:
            print(f"  > {ip}")
            
    else:
        print(f"Failed to reach GitHub. Status Code: {response.status_code}")

except Exception as e:
    print(f"Connection Error: {e}")

In [ ]:
# --- TESTING STEP 3: 

import requests

github_urls = [
    "https://raw.githubusercontent.com/O-X-L/risk-db-lists/refs/heads/main/ip/kind_proxy.txt", 
    "https://raw.githubusercontent.com/O-X-L/risk-db-lists/refs/heads/main/ip/kind_education.txt", 
]

master_data = {}

# Using a Session is faster for multiple requests to the same website
session = requests.Session()

print("Connection established. Storing the Github source to master_data...")

for url in github_urls:
    name = url.split('/')[-1]
    print("Reading " + name + "...", end=" ", flush=True) # Tells you which one it's doing NOW
    
    try:
        # We use a shorter timeout to skip 'dead' links faster
        response = session.get(url, timeout=10)
        
        if response.status_code == 200:
            master_data[name] = response.text.splitlines()
            print("Done! (" + str(len(master_data[name])) + " IPs)")
        else:
            print("Failed (Status " + str(response.status_code) + ")")
            
    except Exception as e:
        print("Skipped due to error.")

print("-" * 30)
print("All files loaded.")

In [ ]:
# --- STEP 3: FETCH AND STORE EVERY GITHUB SOURCE IN MASTER DATA.---

import requests

# Add all the filenames you saw in the GitHub folder here
files_to_fetch = [
    # "kind_crawler.txt",
    # "kind_dynamic.txt",
    # "kind_education.txt",
    # "kind_hosting.txt",
    # "kind_isp.txt",
    # "kind_maybe_hacked.txt",
    # "kind_proxy.txt",
    # "kind_scanner.txt",
    # "kind_tor.txt",
    # "kind_vpn.txt",
    # "top_1000.txt",
    # "top_10000.txt",
    # "top_100000.txt",
    # "main.txt",
    #"exits-v4.txt",
    #"exits-v6.txt",
    "malicious_ip_list.txt"

]

# Add the URL here. Use AI to fix your URL directories
# base_url = "https://raw.githubusercontent.com/O-X-L/risk-db-lists/main/ip/"  
# base_url = "https://raw.githubusercontent.com/sefinek/Malicious-IP-Addresses/main/lists/"
# base_url = "https://raw.githubusercontent.com/Enkidu-6/tor-relay-lists/main/"
base_url = "https://raw.githubusercontent.com/JotaQC/Malicious-IPs-database/main/"


# 1. INITIALIZE AS DICTIONARY (Curly braces)
master_data = {}

print("--- Starting GitHub Fetch ---")

try:
    for file_name in files_to_fetch:
        # Construct the URL automatically
        full_url = f"{base_url}{file_name}"
        
        response = requests.get(full_url, timeout=15)
        
        if response.status_code == 200:
            # Get IPs from this specific file
            current_file_ips = [line.strip() for line in response.text.splitlines() if line.strip()]
            master_data[file_name] = current_file_ips
            
            print(f"✅ Fetched {len(current_file_ips)} IPs from: {file_name}")
        else:
            print(f"❌ Failed to reach {file_name}. Status: {response.status_code}")

    # --- AFTER FETCHING ALL FILES ---
    # Sort the final combined list
    # 3. CREATE UNIQUE SET FOR FAST LOOKUP
    # This "unpacks" all the dictionary values into one big unique set
    github_ips = set().union(*master_data.values())

    print("-" * 30)
    print(f"TOTAL UNIQUE IPs STORED: {len(github_ips)}")
    print("-" * 30)

except Exception as e:
    print(f"Connection Error: {e}")

--- Starting GitHub Fetch ---
✅ Fetched 694 IPs from: malicious_ip_list.txt
------------------------------
TOTAL UNIQUE IPs STORED: 694
------------------------------


In [ ]:
# --- STEP 4: THROUGH MASTER DATA, SEARCH OUR IP LINE BY LINE AND MATCH IT .---

# 1. Safety Check: Did we download from GitHub yet?
if not master_data:
    print("ERROR: The Master List is empty!")
    print("Please run the GitHub Connection first.")
else:
    print("Master List is loaded. Starting check...")
    print("-" * 50)

    # ADJUSTED LINE: Combine all dictionary values (IP lists) into one set
    github_ips = set().union(*master_data.values())

    # 3. Read your local file
    try:
        with open('test1_DataOutbound.txt', 'r', encoding='utf-8') as f:
            my_ips = [line.strip() for line in f if line.strip()]
            
        # 4. The Matching Loop
        for ip in my_ips:
            if ip in github_ips:
                # Find the specific GitHub filename(s)
                sources = [name for name, list_ips in master_data.items() if ip in list_ips]
                # Status at the end. Showing the source file(s) where the IP was found
                print(f"IP: {ip} | Source: {', '.join(sources)} | STATUS: YELLOW")
            else:
                # Status at the end
                print(f"IP: {ip} | STATUS: GREEN")
                
    except FileNotFoundError:
        print("Error: 'test1_DataOutbound.txt' not found in this folder.")

print("-" * 50)
print("Lookup Complete.")

Master List is loaded. Starting check...
--------------------------------------------------
IP: 8.8.8.8 | STATUS: GREEN
IP: 8.8.4.4 | STATUS: GREEN
IP: 58.26.39.135 | STATUS: GREEN
IP: 168.149.132.144 | STATUS: GREEN
IP: 168.149.132.128 | STATUS: GREEN
IP: 168.149.132.160 | STATUS: GREEN
IP: 168.149.132.112 | STATUS: GREEN
IP: 34.120.208.123 | STATUS: GREEN
IP: 2400:7400:4a:2410:10:38:238:9 | STATUS: GREEN
IP: 2400:7400:4a:2410:10:38:238:82 | STATUS: GREEN
IP: 142.250.193.234 | STATUS: GREEN
IP: 172.217.25.138 | STATUS: GREEN
IP: 172.217.25.42 | STATUS: GREEN
IP: 142.251.223.42 | STATUS: GREEN
IP: 172.217.25.234 | STATUS: GREEN
IP: 172.217.25.74 | STATUS: GREEN
IP: 172.217.25.106 | STATUS: GREEN
IP: 172.217.27.10 | STATUS: GREEN
IP: 172.217.25.10 | STATUS: GREEN
IP: 162.159.140.167 | STATUS: GREEN
IP: 172.66.0.165 | STATUS: GREEN
IP: 172.217.26.202 | STATUS: GREEN
IP: 172.217.26.138 | STATUS: GREEN
IP: 172.217.27.42 | STATUS: GREEN
IP: 172.217.26.170 | STATUS: GREEN
IP: 34.102.128.190 

In [ ]:
%pip install pandas openpyxl

In [ ]:
# --- STEP 5: DOWNLOAD FILE.XLSX---
import pandas as pd
# 1. Safety Check: Is the data loaded from GitHub?
if not master_data:
    print("ERROR: Master Data is empty. Please run your GitHub Connection cell first!")
else:
    # 2. Setup the comparison set
    all_bad_ips = set().union(*master_data.values())
    # 3. Read your local file
    try:
        with open('test1_DataOutbound.txt', 'r', encoding='utf-8') as f:
            my_ips = [line.strip() for line in f if line.strip()]

        # 4. Create the data for the Excel table
        rows = []
        for ip in my_ips:
            if ip in all_bad_ips:
                sources = [name for name, list_ips in master_data.items() if ip in list_ips]
                rows.append({
                    "test1_DataOutbound.txt": ip,
                    "Master Data": ", ".join(sources),
                    "STATUS": "YELLOW"
                })
            else:
                rows.append({
                    "test1_DataOutbound.txt": ip,
                    "Master Data": "None",
                    "STATUS": "GREEN"
                })
        # 5. Convert to Excel using Pandas
        df = pd.DataFrame(rows)
        output_name = "IP_verifyGithub.xlsx"
        df.to_excel(output_name, index=False)

        print(f"Success! Your report is ready: {output_name}")
        display(df.head(10)) # Shows the first 10 rows in your notebook

    except FileNotFoundError:
        print("ERROR: Could not find 'test1_DataOutbound.txt'.")

Success! Your report is ready: IP_Security_Report.xlsx


,test1_DataOutbound.txt,Master Data,STATUS
0,8.8.8.8,None,GREEN
1,8.8.4.4,None,GREEN
2,58.26.39.135,None,GREEN
3,168.149.132.144,None,GREEN
4,168.149.132.128,None,GREEN
5,168.149.132.160,None,GREEN
6,168.149.132.112,None,GREEN
7,34.120.208.123,None,GREEN
8,2400:7400:4a:2410:10:38:238:9,None,GREEN
9,2400:7400:4a:2410:10:38:238:82,None,GREEN
